# RealMLP at 700,000 training rows

Fits the GPU neural baseline with the signed validation set used for model selection.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
EXPERIMENT_NAME='lending_club_900k'
FORCE_REBUILD_SPLIT=False
FORCE_RERUN=False
RUN_TESTS=True
# ==================================================================


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive'); repo=Path('/content/tfm-credit-benchmark')
if (repo/'.git').is_dir(): subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH],check=True)
else: subprocess.run(['git','clone','--branch',GITHUB_BRANCH,'--single-branch',GITHUB_REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
print('Repository commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
import json
from tfmbench.config import BenchmarkConfig
from tfmbench.resources import require_cuda
from tfmbench.split import verify_manifest
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
print(json.dumps(require_cuda('RealMLP'),indent=2))
config=BenchmarkConfig(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME); config.validate()
if FORCE_REBUILD_SPLIT:
    from tfmbench.data import prepare_data
    prepare_data(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME,force_rebuild_split=True)
manifest=verify_manifest(config.shared_dir,config); print('Verified manifest:',manifest['manifest_hash'])


In [ ]:
from tfmbench.run import run_method
table=run_method('realmlp',700_000,DRIVE_OUTPUT_DIR,EXPERIMENT_NAME,force_rerun=FORCE_RERUN)


In [ ]:
display(table)
print('RealMLP results:',config.experiment_root/'realmlp'/'n700000')
